# Experimento 4: Topicos por familia politica

Analisis de prevalencia de topicos sustantivos segun familia politica del orador.
Incluye evaluacion de calidad del JOIN orador-legislador.


## 1. Imports y configuracion


In [23]:
import pandas as pd
import numpy as np
import json
import unicodedata
import re
import plotly.express as px
from pathlib import Path

JSONL_PATH = Path(r'C:\Users\Saroka\Documents\GitHub\repositorios\argentine-deputies-discursive-distance\data\processed\modeling_corpus\documents.jsonl')
DATA_DIR   = Path(r'C:\Users\Saroka\Documents\GitHub\repositorios\argentine-deputies-discursive-distance\data\qa\topic_modeling\selected_nmf_k024_v1')
HIST_PATH  = Path(r'C:\Users\Saroka\Documents\GitHub\repositorios\nlp-sesiones-diputados\data\diputados_historial_limpio.csv')
BLOC_PATH  = Path(r'C:\Users\Saroka\Documents\GitHub\repositorios\nlp-sesiones-diputados\data\bloques_familia_politica.csv')

TOPICOS_SUSTANTIVOS = {
    1:  'Derecho penal y codigos procesales',
    4:  'Justicia y poder judicial',
    5:  'Provincias, Buenos Aires y territorio federal',
    8:  'Desarrollo productivo y economia',
    9:  'Derechos, genero y salud reproductiva',
    10: 'Trabajo y empleo',
    16: 'Poderes del ejecutivo, DNU y autoridad constitucional',
    17: 'Presupuesto, gasto e inflacion',
    18: 'Jubilaciones y seguridad social',
    19: 'Deuda, FMI y politica financiera',
    20: 'Impuestos y politica fiscal',
}
TOPICOS_IDS = list(TOPICOS_SUSTANTIVOS.keys())
ORDEN_TOPICOS = [TOPICOS_SUSTANTIVOS[i] for i in TOPICOS_IDS]


## 2. Cargar turnos del corpus


In [24]:
turns = {}
with open(JSONL_PATH, encoding='utf-8') as f:
    for line in f:
        doc = json.loads(line)
        key = (doc['source_record_id'], doc['turn_index'])
        if key not in turns:
            turns[key] = {
                'source_record_id': doc['source_record_id'],
                'turn_index': doc['turn_index'],
                'normalized_label': doc['normalized_label'],
                'speaker_label': doc['speaker_label'],
                'session_date': pd.to_datetime(doc['session_date']),
                'year': doc['year'],
            }

df_turns = pd.DataFrame(turns.values())
print(f'Total turnos unicos: {len(df_turns):,}')
print(f'Labels unicos: {df_turns["normalized_label"].nunique():,}')


Total turnos unicos: 38,774
Labels unicos: 1,373


## 3. Normalizar historial de diputados


In [25]:
def norm(texto):
    if pd.isna(texto): return ''
    nfkd = unicodedata.normalize('NFKD', str(texto))
    return ''.join(c for c in nfkd if not unicodedata.combining(c)).upper().strip()

hist = pd.read_csv(HIST_PATH)
hist['apellido_norm'] = hist['apellido'].apply(norm)
hist['nombre_norm']   = hist['nombre'].apply(norm)
hist['bloque_inicio'] = pd.to_datetime(hist['bloque_inicio'])
hist['bloque_fin']    = pd.to_datetime(hist['bloque_fin'])

bloques = pd.read_csv(BLOC_PATH)
bloques['bloque_norm'] = bloques['bloque'].apply(norm)
hist = hist.merge(bloques[['bloque', 'familia_politica']], on='bloque', how='left')
print(f'Registros historial: {len(hist):,}')


Registros historial: 2,147


## 4. JOIN con clasificacion

Logica:
1. Filtrar candidatos por apellido y fecha exacta de sesion (bloque_inicio <= session_date <= bloque_fin)
2. Deduplicar por nombre (una persona puede tener 2 filas si cambio de bloque)
3. Si el label tiene inicial entre parentesis, usarla para desambiguar
4. Clasificar en: match_unico / ambiguo / sin_match


In [26]:
def extraer_apellido_inicial(label):
    match = re.match(r'^(.+?)\s*\(([^)]+)\)\s*$', label)
    if match:
        return match.group(1).strip(), norm(match.group(2).replace('.','').replace(' ',''))
    return label.strip(), None

def clasificar(label, session_date, hist_df):
    apellido_base, inicial = extraer_apellido_inicial(label)
    # Filtrar por apellido y fecha exacta de sesion
    cands = hist_df[
        (hist_df['apellido_norm'] == apellido_base) &
        (hist_df['bloque_inicio'] <= session_date) &
        (hist_df['bloque_fin']    >= session_date)
    ]
    # Deduplicar: una persona puede tener varios mandatos
    cands = cands.drop_duplicates(subset=['apellido_norm', 'nombre_norm'])
    if len(cands) == 0:
        return 'sin_match', None
    # Intentar desambiguar con inicial
    if inicial and len(cands) > 1:
        filtrado = cands[cands['nombre_norm'].str.replace(' ','').str.startswith(inicial)]
        if len(filtrado) == 1:
            return 'match_unico', filtrado.iloc[0]
    if len(cands) == 1:
        return 'match_unico', cands.iloc[0]
    return 'ambiguo', cands

resultados = []
for _, row in df_turns.iterrows():
    cat, info = clasificar(row['normalized_label'], row['session_date'], hist)
    resultados.append({
        'source_record_id':  row['source_record_id'],
        'turn_index':        row['turn_index'],
        'normalized_label':  row['normalized_label'],
        'session_date':      row['session_date'],
        'year':              row['year'],
        'categoria':         cat,
        'bloque':            info['bloque'] if cat == 'match_unico' else None,
        'familia_politica':  info['familia_politica'] if cat == 'match_unico' else None,
        'nombre_legislador': info['nombre_norm'] if cat == 'match_unico' else None,
    })

df_result = pd.DataFrame(resultados)
print('Listo.')


Listo.


## 5. Calidad del JOIN


In [27]:
conteo = df_result['categoria'].value_counts()
pct    = df_result['categoria'].value_counts(normalize=True).mul(100).round(1)
resumen = pd.DataFrame({'n': conteo, '%': pct})
print('=== CALIDAD DEL JOIN ===')
print(resumen.to_string())


=== CALIDAD DEL JOIN ===
                 n     %
categoria               
match_unico  32480  83.8
sin_match     3465   8.9
ambiguo       2829   7.3


In [28]:
print('=== EJEMPLOS MATCH UNICO ===')
print(df_result[df_result['categoria']=='match_unico']
      [['normalized_label','session_date','nombre_legislador','bloque','familia_politica']]
      .head(10).to_string(index=False))


=== EJEMPLOS MATCH UNICO ===
normalized_label session_date nombre_legislador                                                      bloque       familia_politica
      LOSPENNATO   2021-12-21            SILVIA                                                         pro         centro_derecha
       CASARETTO   2021-12-21     MARCELO PABLO                                             frente de todos peronismo_kirchnerismo
          CAMANO   2021-12-21          GRACIELA                                        identidad bonaerense     provincial_federal
         DEL PLA   2021-12-21            ROMINA partido obrero frente de izquierda y de trabajadores unidad              izquierda
         DEL PLA   2021-12-21            ROMINA partido obrero frente de izquierda y de trabajadores unidad              izquierda
           VILCA   2021-12-21         ALEJANDRO              frente de izquierda y de trabajadores - unidad              izquierda
          CALIVA   2021-12-21      LIA VERONICA       

In [29]:
print('=== EJEMPLOS AMBIGUOS ===')
ambiguos = df_result[df_result['categoria']=='ambiguo']
print(ambiguos[['normalized_label','session_date']]
      .value_counts().head(20).to_string())


=== EJEMPLOS AMBIGUOS ===
normalized_label  session_date
MARTINEZ (D.)     2019-12-19      38
IBARRA            2011-11-30      27
ROSSI (A. O.)     2009-08-12      23
MARTINEZ (G. P.)  2024-04-29      21
MOREAU            2020-11-30      20
ROSSI (A. O.)     2010-03-17      20
MARTINEZ (G. P.)  2024-06-27      19
                  2024-08-14      18
                  2025-06-04      17
                  2025-12-17      17
                  2025-03-12      16
                  2025-10-08      16
                  2025-02-12      15
                  2024-02-02      15
                  2023-09-19      14
GARCIA (A. F.)    2014-11-12      14
MARTINEZ (G. P.)  2023-12-07      13
                  2022-10-25      13
MOREAU            2024-04-29      13
                  2018-07-04      12


In [30]:
print('=== EJEMPLOS SIN MATCH ===')
sin_match = df_result[df_result['categoria']=='sin_match']
print(sin_match['normalized_label'].value_counts().head(20).to_string())


=== EJEMPLOS SIN MATCH ===
normalized_label
JEFE DE GABINETE DE MINISTROS    682
AGUAD                            142
GINZBURG                          67
AZCOITI                           65
PRESIDENTE DE LA NACION           63
MORANDINI                         61
DE MARCHI                         59
LUSQUINOS                         54
SARGHINI                          54
VELARDE                           53
GALVALISI                         48
ACUNA                             47
AUGSBURGER                        46
DIEZ                              44
CANTERO GUTIERREZ                 44
CIGOGNA                           44
GORBACZ                           43
RAIMUNDI                          41
FERRO                             40
SYLVESTRE BEGNIS                  39


## 6. Analisis de topicos por familia politica

Solo se usan los turnos con match_unico.


In [31]:
# Cargar pesos NMF
meta    = pd.read_csv(DATA_DIR / 'source_turn_metadata.csv')
weights = np.load(DATA_DIR / 'topic_weights.npy')  # (34060, 24)

# Construir df con pesos de topicos sustantivos
df_w = meta[['source_turn_row_index', 'source_record_id', 'turn_index']].copy()
for tid in TOPICOS_IDS:
    df_w[f'topic_{tid}'] = weights[:, tid]

# JOIN con resultados del match
df_analisis = df_w.merge(
    df_result[df_result['categoria'] == 'match_unico']
             [['source_record_id','turn_index','familia_politica','bloque']],
    on=['source_record_id','turn_index'], how='inner'
)
print(f'Turnos con match_unico para analisis: {len(df_analisis):,}')
print()
print('Turnos por familia politica:')
print(df_analisis['familia_politica'].value_counts().to_string())


Turnos con match_unico para analisis: 29,197

Turnos por familia politica:
familia_politica
peronismo_kirchnerismo        9353
centro_derecha                5511
radicalismo                   4749
peronismo_federal             3346
centro_izquierda              1979
provincial_federal            1736
izquierda                     1580
derecha_liberal_libertaria     905
otros                           30
sin_bloque                       8


In [32]:
# Etiquetas legibles para el heatmap
FAMILIA_LABELS = {
    'peronismo_kirchnerismo':     'Peronismo / Kirchnerismo',
    'centro_derecha':             'Centro derecha (PRO / CC)',
    'radicalismo':                'Radicalismo (UCR)',
    'peronismo_federal':          'Peronismo federal',
    'centro_izquierda':           'Centro izquierda',
    'provincial_federal':         'Partidos provinciales',
    'izquierda':                  'Izquierda (FIT)',
    'derecha_liberal_libertaria': 'Derecha / Libertaria (LLA)',
}
ORDEN_FAMILIAS = list(FAMILIA_LABELS.values())

# Excluir otros y sin_bloque
df_analisis = df_analisis[df_analisis['familia_politica'].isin(FAMILIA_LABELS.keys())].copy()
df_analisis['familia_label'] = df_analisis['familia_politica'].map(FAMILIA_LABELS)

# Prevalencia media por familia politica
topic_cols = [f'topic_{tid}' for tid in TOPICOS_IDS]
df_prev = df_analisis.groupby('familia_label')[topic_cols].mean()
df_prev.columns = [TOPICOS_SUSTANTIVOS[int(c.split('_')[1])] for c in df_prev.columns]
df_prev = df_prev.reindex(ORDEN_FAMILIAS).T

print('Prevalencia media por familia politica:')
print(df_prev.round(3).to_string())


Prevalencia media por familia politica:
familia_label                                          Peronismo / Kirchnerismo  Centro derecha (PRO / CC)  Radicalismo (UCR)  Peronismo federal  Centro izquierda  Partidos provinciales  Izquierda (FIT)  Derecha / Libertaria (LLA)
Derecho penal y codigos procesales                                        0.018                      0.023              0.025              0.025             0.024                  0.022            0.019                       0.020
Justicia y poder judicial                                                 0.026                      0.026              0.026              0.025             0.026                  0.024            0.018                       0.021
Provincias, Buenos Aires y territorio federal                             0.032                      0.029              0.030              0.027             0.024                  0.043            0.027                       0.029
Desarrollo productivo y economia    

In [33]:
fig = px.imshow(
    df_prev,
    labels=dict(x='Familia politica', y='Topico', color='Prevalencia'),
    title='Prevalencia de topicos sustantivos por familia politica',
    color_continuous_scale='Blues', aspect='auto', height=550, text_auto='.2%'
)
fig.update_layout(coloraxis_colorbar=dict(tickformat='.0%'), xaxis_tickangle=30)
fig.show()


In [34]:
FIG_DIR = Path('figuras/exp4')
FIG_DIR.mkdir(parents=True, exist_ok=True)
fig.write_html(str(FIG_DIR / 'heatmap_topicos_por_familia_politica.html'))
print('Guardado en', FIG_DIR / 'heatmap_topicos_por_familia_politica.html')


Guardado en figuras\exp4\heatmap_topicos_por_familia_politica.html
